# OrbitsWar — Train Classifier & Regressor Models

Trains all four value function models from Kaggle replay data:

| Model | Script | Output |
|---|---|---|
| 1v1 GBC (classifier) | `classifier/train_from_replays.py` | `classifier/value_fn_1v1.py` |
| 4v4 GBC (classifier) | `heuristic_4v4/train_4v4.py` | `heuristic_4v4/value_fn_4v4.py` |
| 1v1 GBR (regressor)  | `regressor/train_from_replays.py` | `regressor/value_fn_1v1.py` |
| 4v4 GBR (regressor)  | `regressor_4v4/train_from_replays.py` | `regressor_4v4/value_fn_4v4.py` |

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
RUN_VS_BOTS = True   # set False to skip live-game fine-tuning

WORK_DIR = "/kaggle/working/OrbitsWar"

# ── Auto-detect dataset mount path ────────────────────────────────────────────
import os

def find_dataset_path():
    base = "/kaggle/input"
    if not os.path.exists(base):
        return None
    # Walk all subdirectories — Kaggle nests datasets under input/datasets/username/slug/
    for root, dirs, files in os.walk(base):
        dirs.sort()  # deterministic traversal order
        # Our dataset has main.py and a records/ folder at its root
        if "main.py" in files and "records" in dirs:
            print(f"Found OrbitsWar dataset at: {root}")
            return root
    # Not found — print full tree to help diagnose
    print("Did not find main.py + records/ anywhere under /kaggle/input/")
    print("Full tree:")
    for root, dirs, files in os.walk(base):
        depth = root.replace(base, "").count(os.sep)
        if depth > 4:
            continue
        indent = "  " * depth
        print(f"{indent}{os.path.basename(root)}/  files={files[:5]}")
    return None

INPUT_DIR = find_dataset_path()

if INPUT_DIR is None:
    raise RuntimeError(
        "Could not find OrbitsWar dataset. See tree printout above.\n"
        "Make sure you attached the dataset to this notebook (Add data -> Your datasets)."
    )

print(f"\nInput:   {INPUT_DIR}")
print(f"WorkDir: {WORK_DIR}")
print(f"Vs-bots: {RUN_VS_BOTS}")

In [ ]:
# ── Copy dataset to writable working directory ─────────────────────────────────
import shutil, sys

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

shutil.copytree(INPUT_DIR, WORK_DIR)
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

print(f"Copied to {WORK_DIR}")
print("Contents:", sorted(os.listdir(WORK_DIR)))

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────────
!pip install -q kaggle-environments scikit-learn
print("Dependencies ready.")

In [ ]:
# ── Verify replay files ────────────────────────────────────────────────────────
import glob
records = glob.glob("records/*.json")
print(f"Replay files: {len(records)}")
if not records:
    print("WARNING: No replay JSON files found in records/")

enemies = glob.glob("enemy agents/*/main.py") + glob.glob("enemy agents/my/*.py")
enemies = [e for e in enemies if not e.endswith("__init__.py")]
print(f"Enemy agents: {len(enemies)}")
for e in sorted(enemies):
    print(f"  {e}")

---
## Phase 1 — 1v1 Classifier (GBC)
Binary win/loss labels. Trains GLOBAL / EARLY / MID / LATE subsets.  
Output: `classifier/value_fn_1v1.py`

In [ ]:
!python classifier/train_from_replays.py

In [ ]:
p = "classifier/value_fn_1v1.py"
print(f"[OK] {p}  ({os.path.getsize(p)/1024:.1f} KB)" if os.path.exists(p) else f"[FAIL] {p}")

---
## Phase 2 — 4v4 Classifier (GBC)
44-dim features, 4-player-aware balancing.  
Output: `heuristic_4v4/value_fn_4v4.py`

In [ ]:
!python heuristic_4v4/train_4v4.py

In [ ]:
p = "heuristic_4v4/value_fn_4v4.py"
print(f"[OK] {p}  ({os.path.getsize(p)/1024:.1f} KB)" if os.path.exists(p) else f"[FAIL] {p}")

---
## Phase 3 — 1v1 Regressor (GBR)
TD-discounted returns: V≈0.97 when dominating at turn 480, V≈0.50 when neutral at turn 100.  
Output: `regressor/value_fn_1v1.py`

In [ ]:
!python regressor/train_from_replays.py

In [ ]:
p = "regressor/value_fn_1v1.py"
print(f"[OK] {p}  ({os.path.getsize(p)/1024:.1f} KB)" if os.path.exists(p) else f"[FAIL] {p}")

---
## Phase 4 — 4v4 Regressor (GBR)
Same TD approach for FFA. Win baseline ~0.25 (1-in-4).  
Output: `regressor_4v4/value_fn_4v4.py`

In [ ]:
!python regressor_4v4/train_from_replays.py

In [ ]:
p = "regressor_4v4/value_fn_4v4.py"
print(f"[OK] {p}  ({os.path.getsize(p)/1024:.1f} KB)" if os.path.exists(p) else f"[FAIL] {p}")

---
## Phase 5 (Optional) — Fine-tune vs Bots
Plays live games against the enemy pool and retrains on combined data.  
Controlled by `RUN_VS_BOTS` in the config cell.

| Script | Games |
|---|---|
| classifier 1v1  | 16 enemies × 10 = 160 games |
| classifier 4v4  | C(16,3) = 560 unique trios |
| regressor 1v1   | 16 enemies × 10 = 160 games |
| regressor 4v4   | C(16,3) = 560 unique trios |

In [ ]:
if RUN_VS_BOTS:
    print("=== 1v1 Classifier vs Bots ===")
    !python classifier/train_1v1_vs_bots.py --games-per-enemy 10
else:
    print("Skipped.")

In [ ]:
if RUN_VS_BOTS:
    print("=== 4v4 Classifier vs Bots ===")
    !python heuristic_4v4/train_4v4_vs_bots.py --passes 1
else:
    print("Skipped.")

In [ ]:
if RUN_VS_BOTS:
    print("=== 1v1 Regressor vs Bots ===")
    !python regressor/train_vs_bots.py --games-per-enemy 10
else:
    print("Skipped.")

In [ ]:
if RUN_VS_BOTS:
    print("=== 4v4 Regressor vs Bots ===")
    !python regressor_4v4/train_vs_bots.py --passes 1
else:
    print("Skipped.")

---
## Output Summary
Files are copied to `/kaggle/working/` so they appear in the notebook output panel for download.

In [ ]:
import shutil

outputs = [
    ("classifier/value_fn_1v1.py",    "classifier_value_fn_1v1.py"),
    ("heuristic_4v4/value_fn_4v4.py", "heuristic_4v4_value_fn_4v4.py"),
    ("regressor/value_fn_1v1.py",     "regressor_value_fn_1v1.py"),
    ("regressor_4v4/value_fn_4v4.py", "regressor_4v4_value_fn_4v4.py"),
]

print("Output files:")
print("-" * 60)
all_ok = True
for src, dest_name in outputs:
    if os.path.exists(src):
        sz  = os.path.getsize(src) / 1024
        dst = f"/kaggle/working/{dest_name}"
        shutil.copy(src, dst)
        print(f"  [OK]   {src:<42} {sz:>7.1f} KB")
    else:
        print(f"  [MISS] {src}")
        all_ok = False

print("-" * 60)
if all_ok:
    print("All 4 models trained. Download files from the Output panel.")
    print()
    print("Local placement after download:")
    print("  classifier_value_fn_1v1.py     -> classifier/value_fn_1v1.py")
    print("  heuristic_4v4_value_fn_4v4.py  -> heuristic_4v4/value_fn_4v4.py")
    print("  regressor_value_fn_1v1.py      -> regressor/value_fn_1v1.py")
    print("  regressor_4v4_value_fn_4v4.py  -> regressor_4v4/value_fn_4v4.py")
    print()
    print("Then run: python submission_hybrid.py")
else:
    print("Some models failed — check cell output above.")